# 6 · Model Router: a cheap judge in front of expensive LLMs

## The problem

Most teams send **every** request to their best (most expensive) model. But a big share of real traffic is easy:
*"hi there"*, *"what's the capital of Australia?"*, *"reformat these names"*. Paying frontier-model prices for those is like
sending every patient in a hospital straight to the head surgeon.

A **model router** fixes this. It looks at each request first and sends it to the **cheapest model that can handle it well**.

## The catch, and why Jev

The router itself runs on **every** request, so it must be much cheaper and faster than what it saves. If you used an LLM
to decide which LLM to call, you'd pay for two LLM calls instead of one.

Jev is a **System One** model: it doesn't write text, it returns **typed judgments with calibrated probabilities**, in about
100 ms, for a fraction of a cent. That makes it a good router.

> **Jev decides, OpenAI runs.**

## What we'll build

```
user prompt ──► Jev: 3 judgments in ONE call ──► plain-Python policy ──► chosen OpenAI model ──► answer
                 • difficulty (Score 0..3)        • difficulty → tier
                 • kind       (Choice)            • + 3 safety overrides
                 • high stakes (Noul)
```

## What you'll learn
- The three Jev question types (`Score`, `Choice`, `Noul`) and how to read their answers.
- The core design idea: **the model measures, your code decides**. Every threshold sits in code where you can read, test and change it.
- How to estimate what a router saves.

> **How to run this notebook**
> 1. Pick the **`.venv`** kernel (top right in VS Code, *Kernel → Change kernel* in Jupyter).
> 2. Run the cells **top to bottom** (or *Run All*). Every cell builds on the ones above it.
> 3. If you see `NameError: name '...' is not defined`, the kernel was restarted (switching kernels does this) and forgot the earlier cells. Click **Run All** or *Run All Above*, then continue.

## Step 0 · Setup

This cell:
- loads the API keys from the `.env` file at the repo root (`TYPESAFE_API_KEY` for Jev, `OPENAI_API_KEY` for OpenAI),
- creates **two clients**: `jev` makes the routing *decision*, `oai` runs the chosen model,
- imports the three Jev question types we'll use.

In [1]:
import os
import time
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from typesafe_sdk import TypeSafeClient, Noul, Score, Choice

# Look for a .env file in this folder or any parent folder
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").exists():
        load_dotenv(folder / ".env")
        break

jev = TypeSafeClient()   # the judge: fast, cheap, typed answers
oai = OpenAI()           # the workers: the models that actually write the answer

print("clients ready")

clients ready


## Step 1 · The model ladder

Three OpenAI models from cheapest to most capable. `price` is a **relative cost per call** that we use to compare
routing strategies. It's illustrative, not an official price list. What matters is the ratio: the frontier model costs
**25×** the small one.

In [2]:
TIERS = {
    "small":    {"model": "gpt-5.4-nano", "price": 0.05, "desc": "fast, cheap"},
    "mid":      {"model": "gpt-5.4-mini", "price": 0.25, "desc": "balanced"},
    "frontier": {"model": "gpt-5.4",      "price": 1.25, "desc": "deep reasoning"},
}

for name, t in TIERS.items():
    print(f"{name:<9} {t['model']:<13} relative cost {t['price']:.2f}  ({t['desc']})")

small     gpt-5.4-nano  relative cost 0.05  (fast, cheap)
mid       gpt-5.4-mini  relative cost 0.25  (balanced)
frontier  gpt-5.4       relative cost 1.25  (deep reasoning)


## Step 2 · Define the questions Jev will answer

We ask Jev **three questions about every prompt, in a single request**. Each one uses a different question type:

| Key | Type | What it returns | Why the router needs it |
|---|---|---|---|
| `difficulty` | `Score` | `.score`, a number from 0 to 3 (one step per rubric level), plus `.confidence` | the main signal for choosing a tier |
| `kind` | `Choice` | `.choice` (the winning label) and `.probabilities` (one per label) | math/logic tasks deserve at least a mid model |
| `high_stakes` | `Noul` | `.noul`, the probability that the answer is "yes" (0 to 1) | costly mistakes go to the best model regardless |

**How a `Score` works:** the `criteria` list is an ordered rubric. Level 0 is the first line, level 3 the last. Jev
returns a *continuous* position on that scale, so 2.4 means "between Hard and Expert, closer to Hard".

This cell only **defines** the questions. Nothing is sent to Jev yet.

In [3]:
ROUTER_QUESTIONS = {
    "difficulty": Score(
        instructions="How much reasoning does answering this request well require?",
        criteria=[
            "Trivial: a lookup, a greeting, or a one-line fact",          # 0
            "Moderate: a short explanation or a small, well-scoped task",  # 1
            "Hard: multi-step work, careful code, or nuanced judgment",    # 2
            "Expert: deep multi-step reasoning, proofs, or tricky debugging",  # 3
        ],
    ),
    "kind": Choice(
        instructions="What kind of request is this?",
        criteria={
            "factual_lookup": "Asking for a known fact or definition",
            "simple_rewrite": "Rephrase, summarize, or reformat supplied text",
            "code": "Write, fix, or explain code",
            "reasoning_math": "Logic, math, or multi-step problem solving",
            "creative": "Open-ended writing or brainstorming",
        },
    ),
    "high_stakes": Noul(
        instructions="Would a wrong answer here be costly, unsafe, or hard to undo?",
    ),
}
print("defined", len(ROUTER_QUESTIONS), "questions:", list(ROUTER_QUESTIONS))

defined 3 questions: ['difficulty', 'kind', 'high_stakes']


## Step 3 · Ask once and read the raw answer

`jev.system_one(state=..., questions=...)` sends the prompt (the **state**) plus all three questions in **one** call.
The answers come back sorted by type: `r.scores[...]`, `r.choices[...]`, `r.nouls[...]`.

In [4]:
r = jev.system_one(state="Prove that the square root of 2 is irrational.", questions=ROUTER_QUESTIONS)

d = r.scores["difficulty"]
print(f"difficulty : {d.score:.2f} / 3   (confidence {d.confidence:.2f})")

k = r.choices["kind"]
print(f"kind       : {k.choice}")
for label, p in sorted(k.probabilities.items(), key=lambda kv: -kv[1]):
    print(f"               {label:<15} {p:.2f}")

print(f"high_stakes: {r.nouls['high_stakes'].noul:.2f}")
print(f"tokens     : {r.usage.input_tokens}")

difficulty : 2.44 / 3   (confidence 0.45)
kind       : reasoning_math
               reasoning_math  1.00
               factual_lookup  0.00
               simple_rewrite  0.00
               code            0.00
               creative        0.00
high_stakes: 0.08
tokens     : 519


**What to notice**
- Difficulty lands between *Hard* and *Expert*: a proof needs real reasoning.
- `kind` is a full probability distribution, not just one label. You can use the runner-up too.
- `confidence` on the difficulty can be fairly low even when the score is high. Jev is telling you how *sure* it is, and
  our policy will use that.
- It took **one** request for all three answers.

## Step 4 · The routing policy, in plain Python

This is the most important cell in the notebook. Jev only produced **numbers**. Every *decision* happens here, in code you
can read, unit-test and change.

**Base rule: difficulty picks the tier**

| difficulty score | tier |
|---|---|
| below 1.0 (trivial) | `small` |
| 1.0 to 2.2 (moderate to hard) | `mid` |
| 2.2 and above (hard to expert) | `frontier` |

**Then three safety overrides. They only ever move a request *up*, never down:**
1. **Unsure → no cheap route.** If Jev isn't confident about the difficulty (`confidence < 0.55`) and we were about to use `small`, use `mid` instead. Being wrong in the cheap direction is the costly mistake.
2. **Math or logic → at least `mid`.** Tiny models are weakest at multi-step reasoning.
3. **High stakes → `frontier`.** If a wrong answer would be costly (`high_stakes > 0.7`), pay for the best model.

`reasons` records *why* each decision was made, so you can always explain a routing choice.

In [5]:
CONFIDENT = 0.55       # below this, don't trust a "trivial" verdict
SMALL_MAX = 1.0        # difficulty below this -> small
MID_MAX = 2.2          # difficulty below this -> mid, otherwise frontier
STAKES_MAX = 0.7       # high_stakes above this -> frontier


def choose_tier(prompt, client):
    if "ROUTER_QUESTIONS" not in globals():
        raise RuntimeError("ROUTER_QUESTIONS is missing: the kernel restarted. Use 'Run All' (or run the cells above) first.")

    r = client.system_one(state=prompt, questions=ROUTER_QUESTIONS)
    diff = r.scores["difficulty"]
    kind = r.choices["kind"]
    stakes = r.nouls["high_stakes"].noul

    # Base rule: difficulty picks the tier
    if diff.score < SMALL_MAX:
        tier = "small"
    elif diff.score < MID_MAX:
        tier = "mid"
    else:
        tier = "frontier"
    reasons = [f"difficulty {diff.score:.2f}"]

    # Override 1: unsure -> no cheap route
    if diff.confidence < CONFIDENT and tier == "small":
        tier = "mid"
        reasons.append(f"unsure ({diff.confidence:.2f}), no cheap route")
    # Override 2: reasoning tasks get at least mid
    if kind.choice == "reasoning_math" and tier == "small":
        tier = "mid"
        reasons.append("reasoning task, floor at mid")
    # Override 3: high stakes -> frontier
    if stakes > STAKES_MAX and tier != "frontier":
        tier = "frontier"
        reasons.append(f"high stakes {stakes:.2f}")

    return {"tier": tier, "difficulty": diff.score, "confidence": diff.confidence,
            "kind": kind.choice, "stakes": stakes, "reasons": reasons}


print(choose_tier("hi there", jev))

{'tier': 'small', 'difficulty': 0.0, 'confidence': 1.0, 'kind': 'creative', 'stakes': 0.04, 'reasons': ['difficulty 0.00']}


## Step 5 · Route a mixed batch, and compute the savings

No OpenAI calls yet, just the routing decisions. Then we compare two bills:
- **always-frontier:** every prompt goes to the top model (what most teams do),
- **with router:** each prompt goes to the tier the policy chose.

In [6]:
PROMPTS = [
    "What's the capital of Australia?",
    "Reformat 'john smith, jane doe' to Title Case.",
    "Prove that the square root of 2 is irrational.",
    "Write a haiku about a tired Monday.",
    "A payment webhook fires twice per event intermittently. What are the top 3 causes?",
    "I'm about to run a database migration on production. Is it safe to drop the users.email index?",
    "hi there",
]

routed_cost = 0.0
for p in PROMPTS:
    d = choose_tier(p, jev)
    routed_cost += TIERS[d["tier"]]["price"]
    print(f"{d['tier']:<9} {p[:62]:<62}")
    print(f"          why: {'; '.join(d['reasons'])}")

always_frontier = TIERS["frontier"]["price"] * len(PROMPTS)
saving = (1 - routed_cost / always_frontier) * 100
print(f"\nrelative bill  always-frontier {always_frontier:.2f}   with router {routed_cost:.2f}   -> {saving:.0f}% cheaper")

small     What's the capital of Australia?                              
          why: difficulty 0.00


mid       Reformat 'john smith, jane doe' to Title Case.                
          why: difficulty 0.50; unsure (0.50), no cheap route


frontier  Prove that the square root of 2 is irrational.                
          why: difficulty 2.40


small     Write a haiku about a tired Monday.                           
          why: difficulty 0.90


mid       A payment webhook fires twice per event intermittently. What a
          why: difficulty 1.35


frontier  I'm about to run a database migration on production. Is it saf
          why: difficulty 1.85; high stakes 0.89


small     hi there                                                      
          why: difficulty 0.00

relative bill  always-frontier 8.75   with router 3.15   -> 64% cheaper


**What to notice**
- Greetings and facts go to `small`, the proof to `frontier`, the debugging question to `mid` or higher.
- The production-migration question shows **override 3** at work: it may not be hard, but a wrong answer is expensive, so it goes to the best model.
- The saving depends entirely on your traffic mix. Measure it on *your* real prompts, and remember the router call itself has a (small) cost.

## Step 6 · Route, then actually run the chosen model

`run()` is the full router: ask Jev for the tier, then call that OpenAI model and time it. This is the function a real
app would call instead of calling OpenAI directly.

In [7]:
def run(prompt, client, oai):
    d = choose_tier(prompt, client)
    d["model"] = TIERS[d["tier"]]["model"]

    t0 = time.perf_counter()
    resp = oai.chat.completions.create(
        model=d["model"],
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=2000,
    )
    d["latency_ms"] = (time.perf_counter() - t0) * 1000
    d["answer"] = (resp.choices[0].message.content or "").strip().replace("\n", " ")
    return d


d = run("What's the capital of Australia?", jev, oai)
print(f"{d['tier']} -> {d['model']} in {d['latency_ms']:.0f} ms")
print("answer:", d["answer"])

small -> gpt-5.4-nano in 803 ms
answer: The capital of Australia is **Canberra**.


## Step 7 · Give it a face

Everything from here on is **presentation only**: a small style kit, a text box wired to `run()` with `ipywidgets`, and
an HTML card showing the tier, the model, the reasons and the answer. The routing logic above is untouched.

In [8]:
import ipywidgets as widgets
from IPython.display import HTML, display

INK, SLATE, SAGE, AMBER, RED, LINE = "#1F1E1D", "#5E6B7A", "#1F7A3D", "#C98A00", "#C0392B", "#DEDACE"
CARD = f"background:#FAF9F5;border:1px solid {LINE};border-radius:10px;padding:14px 16px;margin:8px 0;"
BADGE = "display:inline-block;padding:2px 10px;border-radius:12px;color:white;font-weight:600;font-size:12px;"
TIER_COLOR = {"small": SAGE, "mid": AMBER, "frontier": RED}


def box(inner):
    return f'<div style="font-family:-apple-system,Segoe UI,sans-serif;color:{INK};max-width:820px;">{inner}</div>'


def router_html(prompt):
    d = run(prompt, jev, oai)
    col = TIER_COLOR[d["tier"]]
    return box(f'<div style="{CARD}border-left:5px solid {col};">'
               f'<div style="margin-bottom:6px;"><b>{prompt}</b></div>'
               f'<span style="{BADGE}background:{col};">{d["tier"].upper()}</span> '
               f'<code>{d["model"]}</code> &nbsp; {d["latency_ms"]:.0f} ms &nbsp; '
               f'<span style="color:{SLATE};">difficulty {d["difficulty"]:.2f} · {d["kind"]} · stakes {d["stakes"]:.2f}</span>'
               f'<div style="margin-top:8px;color:{SLATE};font-size:12px;">why: {"; ".join(d["reasons"])}</div>'
               f'<div style="margin-top:8px;padding:8px;background:#F0EEE6;border-radius:6px;">{d["answer"][:500]}</div></div>')


def text_app(placeholder, render, button="Run"):
    inp = widgets.Text(placeholder=placeholder, continuous_update=False, layout=widgets.Layout(width="68%"))
    btn = widgets.Button(description=button, button_style="warning")
    out = widgets.Output()

    def go(_):
        with out:
            out.clear_output()
            if inp.value.strip():
                display(HTML(box("<i>routing...</i>")))
                html = render(inp.value.strip())
                out.clear_output(wait=True)
                display(HTML(html))

    btn.on_click(go)
    inp.observe(go, names="value")   # pressing Enter submits
    display(widgets.HBox([inp, btn]), out)


display(HTML(router_html("Prove that the square root of 2 is irrational.")))

## Step 8 · Try it yourself

Type a prompt and press **Enter** or click the button (needs a live kernel). Good ones to try in a workshop:
- `hi!` → should be **small**
- `Explain recursion to a 10-year-old` → probably **mid**
- `Find the race condition: two threads increment a shared counter without a lock` → **mid** or **frontier**
- `What dose of ibuprofen is safe for my toddler?` → watch the **high stakes** override kick in

In [9]:
text_app("ask anything, e.g. 'write a limerick' or 'debug this race condition'", router_html, "Route + Run")

Output()

## Recap and exercises

**Key ideas**
1. A router must be **much cheaper than what it saves**. A System One judge (~100 ms, sub-cent) makes the economics work.
2. **The model measures, the code decides.** Jev returned three numbers; tiers, thresholds and overrides are ordinary Python.
3. **Use confidence.** A low-confidence "trivial" verdict is a reason *not* to take the cheap path.
4. Overrides should push towards safety (up a tier), never down.

**Exercises**
- Change `MID_MAX` from 2.2 to 1.8 and re-run Step 5. How does the bill change?
- Add a rule: `kind == "code"` gets at least `mid`.
- Add a fourth tier, e.g. a *rule-based* tier that answers greetings without calling any model.
- Collect 20 of your own real prompts, label the tier you'd want, and measure how often the router agrees.